In [1]:
#!/usr/bin/python
# -*- coding: utf-8 -*-

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import pdist, squareform
from pymatgen.core import Structure
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer

# ==========================================
# 1. 核心特征计算逻辑 (保持原样)
# ==========================================

def get_element_symbol(species_str):
    return "".join([c for c in species_str if c.isalpha()])

def compute_CE_feature(center_atom, neighbors, elements_properties):
    center_species = center_atom['species']
    
    # 错误处理：防止表格里没有该元素
    if center_species not in elements_properties.columns:
        raise ValueError(f"元素属性表中缺少元素: {center_species}")
        
    center_prop = elements_properties[center_species]
    center_indexed = pd.Series(center_prop.values, index=[f"C_{attr}" for attr in center_prop.index])

    # 这里的权重逻辑：依然使用 1/d
    # 注意：Shell方法可能会有很远的原子，距离大权重小，符合物理规律
    weights = np.array([1 / n['distance'] if n['distance'] != 0 else 0 for n in neighbors])
    if weights.sum() > 0:
        weights /= weights.sum()
    else:
        weights = np.zeros_like(weights)

    env_prop = pd.Series(0, index=center_prop.index)
    for n, w in zip(neighbors, weights):
        n_species = n['species']
        if n_species in elements_properties.columns:
             env_prop += elements_properties[n_species] * w
        else:
            # 如果邻居元素不在表中，跳过或报错，这里选择静默跳过但需注意
            pass

    env_indexed = pd.Series(env_prop.values, index=[f"E_{attr}" for attr in env_prop.index])

    diff_prop = (center_prop - env_prop).abs()
    diff_indexed = pd.Series(diff_prop.values, index=[f"D_{attr}" for attr in center_prop.index])

    ratio_prop = center_prop / (env_prop + 1e-6)
    ratio_indexed = pd.Series(ratio_prop.values, index=[f"R_{attr}" for attr in center_prop.index])

    CE_feature = pd.concat([center_indexed, env_indexed, diff_indexed, ratio_indexed])
    return CE_feature

# ==========================================
# 2. 数据加载 (保持原样)
# ==========================================

def load_elements_properties(file_path="/home/gaotianjiao/modnet-master/ext_struc/element_properties/ElementsProperties.xlsx"):
    elements_df = pd.read_excel(file_path, sheet_name=0, index_col=None, header=0)
    elements_list = pd.read_excel(file_path, sheet_name=1, index_col=None, header=0)
    elements = list(elements_list["element"])
    
    # 确保只读取存在的列
    valid_elements = [e for e in elements if e in elements_df.columns]
    df = elements_df.loc[:, valid_elements]
    
    df.dropna(axis=0, how='any', inplace=True)
    df = df.reset_index(drop=True)
    property_names = elements_df["Properties"].tolist()
    return df, property_names

# ==========================================
# 3. [修改核心]：Shell 邻居查找法
# ==========================================

def get_motifs_by_shells(structure, num_shells=3, tolerance=0.01):
    """
    使用“距离壳层法”寻找邻居 (模仿之前的ASE代码逻辑)
    
    Parameters
    ----------
    structure : pymatgen.core.Structure
    num_shells : int
        截断层数，默认取前3层邻居
    tolerance : float
        判断距离是否相等的容差
        
    Returns
    -------
    motifs : list
        符合 extract_CE_features_from_motifs 输入格式的列表
    """
    
    # 使用 SpacegroupAnalyzer 找不对称单元 (Unique Sites)
    sga = SpacegroupAnalyzer(structure)
    symmetrized_structure = sga.get_symmetrized_structure()
    
    motifs = []
    
    # 遍历每一个不等价位点 (Unique Sites)
    for i, group in enumerate(symmetrized_structure.equivalent_indices):
        # 取该组的第一个原子作为代表
        center_index = group[0]
        center_site = structure[center_index]

        all_neighbors = structure.get_neighbors(center_site, r=10.0)
        
        if not all_neighbors:
            continue
            
        # 2. 按距离排序
        all_neighbors.sort(key=lambda x: x[1]) # x[1] is distance
        
        # 3. 距离分层 (Shelling)
        shells = []
        current_shell = []
        
        if all_neighbors:
            current_dist = all_neighbors[0][1]
            
            for neighbor in all_neighbors:
                site, dist = neighbor[0], neighbor[1]
                
                # 如果距离变化超过容差，说明进入了下一层
                if dist - current_dist > tolerance:
                    shells.append(current_shell) # 保存上一层
                    current_shell = []
                    current_dist = dist
                
                # 格式化邻居信息
                current_shell.append({
                    'species': site.specie.symbol,
                    'distance': dist,
                    'index': neighbor[2] # 原始索引
                })
            
            if current_shell:
                shells.append(current_shell)
        
        # 4. 截断：只取前 num_shells 层
        selected_shells = shells[:num_shells]
        
        # 5. 展平列表：把选中的几层合并成一个 neighbors 列表
        final_neighbors = [n for shell in selected_shells for n in shell]
        
        motifs.append({
            'central_atom': {
                'species': center_site.specie.symbol,
                'index': center_index
            },
            'neighbors': final_neighbors,
            'coordination_environment': f"{len(selected_shells)}_shells", # 标记用了几层
            'distorted': False 
        })
        
    return motifs

def extract_CE_features_from_motifs(structure, num_shells=3, elements_properties_file="/home/gaotianjiao/modnet-master/ext_struc/element_properties/ElementsProperties.xlsx"):
    """
    主函数：输入结构 -> 计算基于Shell的CE特征
    """
    # 1. 加载属性
    elements_properties, property_names = load_elements_properties(elements_properties_file)
    
    # 2. [修改] 使用 Shell 方法获取 motifs
    motifs = get_motifs_by_shells(structure, num_shells=num_shells)
    
    features = []

    for motif in motifs:
        center_atom = motif['central_atom']
        # 3. 计算 C, E, D, R 特征
        CE_vec = compute_CE_feature(center_atom=center_atom,
                                    neighbors=motif['neighbors'],
                                    elements_properties=elements_properties)
        features.append({
            'center_index': center_atom['index'],
            'center_species': center_atom['species'],
            'coordination_environment': motif.get('coordination_environment', ''),
            'distorted': motif.get('distorted', False),
            'CE_feature': CE_vec
        })
    
    return features, property_names

# ==========================================
# 5. 后处理与加权 (保持原样)
# ==========================================

def rename_CE_feature_columns(ce_df, property_names):
    n_props = len(property_names)
    C_cols = ["C_" + name for name in property_names]
    E_cols = ["E_" + name for name in property_names]
    D_cols = ["D_" + name for name in property_names] 
    R_cols = ["R_" + name for name in property_names]

    expected_cols = 4 * n_props
    # 弱化检查，防止有些属性因为NaN被丢弃导致列数不匹配
    # if ce_df.shape[1] != expected_cols:
    #     raise ValueError(f"列数不匹配")

    ce_df_renamed = ce_df.copy()
    ce_df_renamed.columns = C_cols + E_cols + D_cols + R_cols
    return ce_df_renamed

def weighted_CE_vector(ce_features, method='hybrid', scale_method='zscore'):
    # 检查是否为空
    if not ce_features:
        return None
        
    all_keys = ce_features[0]['CE_feature'].index
    X = np.array([f['CE_feature'].values for f in ce_features])

    n_motifs = len(ce_features)
    if n_motifs == 1:
        return ce_features[0]['CE_feature']

    if scale_method == 'zscore':
        scaler = StandardScaler()
    elif scale_method == 'minmax':
        from sklearn.preprocessing import MinMaxScaler
        scaler = MinMaxScaler()
    else:
        raise ValueError("scale_method 必须是 'zscore' 或 'minmax'")

    # 注意：如果样本数少于特征数，Scaler可能报错或警告，这里假设特征矩阵正常
    X_scaled = scaler.fit_transform(X)

    var_weights = np.var(X_scaled, axis=1)
    if np.all(var_weights == 0):
        var_weights = np.ones_like(var_weights)

    dist_matrix = squareform(pdist(X_scaled, metric='euclidean'))
    dist_weights = np.mean(dist_matrix, axis=1)

    if method == 'variance':
        weights = var_weights
    elif method == 'distance':
        weights = dist_weights
    elif method == 'hybrid':
        weights = (var_weights + dist_weights) / 2
    else:
        raise ValueError("method 必须是 'variance', 'distance' 或 'hybrid'")

    weights = np.maximum(weights, 1e-8)
    weights /= np.sum(weights)

    weighted_X = X * weights[:, np.newaxis]
    final_vector_values = np.sum(weighted_X, axis=0)

    final_vector = pd.Series(final_vector_values, index=all_keys)
    return final_vector



In [4]:
# ==========================================
# 6. 测试运行脚本
# ==========================================
if __name__ == "__main__":
    # 示例：这就需要你自己提供 CIF 文件路径和 Excel 文件了
    print("代码已更新：现在使用 [Shell/Distance] 逻辑寻找邻居，并计算 [C/E/D/R] 特征。")
    print("请确保目录下有 ElementsProperties.xlsx")

    struc = Structure.from_file("/home/gaotianjiao/modnet-master/ext_struc/perovskite_cif/mp-981250.cif")
    feats, p_names = extract_CE_features_from_motifs(struc, num_shells=3)
    final_vec = weighted_CE_vector(feats)
    df = pd.DataFrame([final_vec])
    df = rename_CE_feature_columns(df, p_names)
    print(df.head())

代码已更新：现在使用 [Shell/Distance] 逻辑寻找邻居，并计算 [C/E/D/R] 特征。
请确保目录下有 ElementsProperties.xlsx


FileNotFoundError: [Errno 2] No such file or directory: '/home/gaotianjiao/modnet-master/ext_struc/pervovskite_cif/mp-981250.cif'

In [5]:
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import pdist, squareform
from pymatgen.core import Structure
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer

# ==========================================
# 1. 核心函数定义 (保持和你主程序一致)
# ==========================================

def load_elements_properties(file_path):
    print(f"-> 正在加载属性表: {file_path}")
    try:
        elements_df = pd.read_excel(file_path, sheet_name=0, index_col=None, header=0)
        elements_list = pd.read_excel(file_path, sheet_name=1, index_col=None, header=0)
    except Exception as e:
        print(f"❌ 属性表读取失败: {e}")
        return None, None

    elements_df.columns = [c.strip() for c in elements_df.columns]
    elements = list(elements_list["element"])
    valid_elements = [e for e in elements if e in elements_df.columns]
    
    df = elements_df.loc[:, valid_elements]
    df.dropna(axis=0, how='any', inplace=True)
    df = df.reset_index(drop=True)
    property_names = elements_df["Properties"].tolist()
    return df, property_names

def get_motifs_by_shells(structure, num_shells=3, tolerance=0.01):
    try:
        sga = SpacegroupAnalyzer(structure)
        symmetrized_structure = sga.get_symmetrized_structure()
        indices_list = symmetrized_structure.equivalent_indices
    except:
        indices_list = [[i] for i in range(len(structure))]

    motifs = []
    for group in indices_list:
        center_index = group[0]
        center_site = structure[center_index]
        # 找邻居
        all_neighbors = structure.get_neighbors(center_site, r=10.0)
        if not all_neighbors: continue
        all_neighbors.sort(key=lambda x: x[1])
        
        shells = []
        current_shell = []
        if all_neighbors:
            current_dist = all_neighbors[0][1]
            for neighbor in all_neighbors:
                site, dist = neighbor[0], neighbor[1]
                if dist - current_dist > tolerance:
                    shells.append(current_shell)
                    current_shell = []
                    current_dist = dist
                current_shell.append({'species': site.specie.symbol, 'distance': dist, 'index': neighbor[2]})
            if current_shell: shells.append(current_shell)
        
        final_neighbors = [n for shell in shells[:num_shells] for n in shell]
        
        motifs.append({
            'central_atom': {'species': center_site.specie.symbol, 'index': center_index},
            'neighbors': final_neighbors,
            'debug_shell_count': len(shells[:num_shells]) # 用于调试打印
        })
    return motifs

def compute_CE_feature(center_atom, neighbors, elements_properties):
    center_species = center_atom['species']
    if center_species not in elements_properties.columns:
        return None 

    center_prop = elements_properties[center_species]
    weights = np.array([1 / n['distance'] if n['distance'] != 0 else 0 for n in neighbors])
    if weights.sum() > 0: weights /= weights.sum()
    
    env_prop = pd.Series(0, index=center_prop.index)
    for n, w in zip(neighbors, weights):
        if n['species'] in elements_properties.columns:
            env_prop += elements_properties[n['species']] * w

    feature_vec = np.concatenate([
        center_prop.values, 
        env_prop.values, 
        np.abs(center_prop.values - env_prop.values), 
        center_prop.values / (env_prop.values + 1e-6)
    ])
    
    idx = [f"{p}_{x}" for p in ['C','E','D','R'] for x in center_prop.index]
    return pd.Series(feature_vec, index=idx)

def weighted_CE_vector(ce_features_list):
    valid = [f for f in ce_features_list if f is not None]
    if not valid: return None
    X = np.array([f.values for f in valid])
    if len(valid) == 1: return valid[0]
    
    try:
        X_scaled = StandardScaler().fit_transform(X)
    except:
        X_scaled = X
        
    var_w = np.var(X_scaled, axis=1)
    if np.all(var_w == 0): var_w = np.ones_like(var_w)
    dist_w = np.mean(squareform(pdist(X_scaled)), axis=1)
    
    w = (var_w + dist_w) / 2
    w = np.maximum(w, 1e-8)
    w /= np.sum(w)
    
    return pd.Series(np.sum(X * w[:, np.newaxis], axis=0), index=valid[0].index)


In [6]:
from tqdm import tqdm
import os
# --- Configuration Paths ---
INPUT_FILE = "/home/gaotianjiao/modnet-master/ext_struc/feature_pervoskite_ef_noce.csv"
OUTPUT_FILE = "/home/gaotianjiao/modnet-master/ext_struc/feature_pervoskite_ef.csv"
CIF_FOLDER = "/home/gaotianjiao/modnet-master/ext_struc/perovskite_cif"
PROP_FILE = "/home/gaotianjiao/modnet-master/ext_struc/element_properties/ElementsProperties.xlsx"

print("=== 🚀 Starting Batch Processing ===")

# 1. Load Properties
props, p_names = load_elements_properties(PROP_FILE)
if props is None:
    print("❌ Failed to load properties. Exiting.")
    exit()
print(f"✅ Properties loaded. Shape: {props.shape}")

# 2. Load Input Data
print(f"-> Reading input file: {INPUT_FILE}")
if INPUT_FILE.endswith('.xlsx'):
    df = pd.read_excel(INPUT_FILE)
else:
    df = pd.read_csv(INPUT_FILE)

print(f"-> Total rows to process: {len(df)}")

# 3. Main Loop
extracted_features = []
error_count = 0

# Using tqdm for progress bar
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing CIFs"):
    
    # Get filename (assuming column exists and already has .cif suffix based on your test)
    cif_name = str(row['cif_file'])
    cif_path = os.path.join(CIF_FOLDER, cif_name)
    
    feature_series = None
    
    if os.path.exists(cif_path):
        try:
            # A. Read Structure
            struct = Structure.from_file(cif_path)
            
            # B. Find Shell Neighbors
            motifs = get_motifs_by_shells(struct, num_shells=3)
            
            # C. Compute Features per Motif
            motif_vectors = []
            for m in motifs:
                vec = compute_CE_feature(m['central_atom'], m['neighbors'], props)
                motif_vectors.append(vec)
            
            # D. Weighted Aggregation
            feature_series = weighted_CE_vector(motif_vectors)
            
        except Exception as e:
            # error_count += 1
            # print(f"Error processing {cif_name}: {e}")
            pass
    else:
        # File not found
        pass
if feature_series is not None:  # <--- 加上这个判断
    extracted_features.append(feature_series)

# 4. Merge and Save
print("\n=== Merging Data ===")
feat_df = pd.DataFrame(extracted_features)

if feat_df.empty or feat_df.isnull().all().all():
    print("❌ Warning: No features were extracted! Please check paths or data.")
else:
    # Reset indices to ensure correct horizontal concatenation
    df.reset_index(drop=True, inplace=True)
    feat_df.reset_index(drop=True, inplace=True)
    
    # Concatenate original data with new features
    final_df = pd.concat([df, feat_df], axis=1)
    
    print(f"-> Saving result to: {OUTPUT_FILE}")
    if OUTPUT_FILE.endswith('.xlsx'):
        final_df.to_excel(OUTPUT_FILE, index=False)
    else:
        feature_series.to_csv(OUTPUT_FILE, index=False)
    
    print("✅ Batch processing complete!")

=== 🚀 Starting Batch Processing ===
-> 正在加载属性表: /home/gaotianjiao/modnet-master/ext_struc/element_properties/ElementsProperties.xlsx
✅ Properties loaded. Shape: (37, 77)
-> Reading input file: /home/gaotianjiao/modnet-master/ext_struc/feature_pervoskite_ef_noce.csv
-> Total rows to process: 810


Processing CIFs:  10%|▉         | 79/810 [00:04<01:04, 11.26it/s]/home/gaotianjiao/anaconda3/lib/python3.8/site-packages/pymatgen/io/cif.py:1168: UserWarning: Issues encountered while parsing CIF: Some fractional coordinates rounded to ideal values to avoid issues with finite precision.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))
Processing CIFs:  90%|█████████ | 730/810 [00:50<00:02, 39.86it/s]/home/gaotianjiao/anaconda3/lib/python3.8/site-packages/pymatgen/io/cif.py:1168: UserWarning: Issues encountered while parsing CIF: Some fractional coordinates rounded to ideal values to avoid issues with finite precision.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))
Processing CIFs: 100%|██████████| 810/810 [00:54<00:00, 14.88it/s]


=== Merging Data ===
-> Saving result to: /home/gaotianjiao/modnet-master/ext_struc/feature_pervoskite_ef.csv
✅ Batch processing complete!


In [7]:
feature_series=feature_series.dropna()

In [8]:
feature_series

C_0       17.991613
C_1       37.268138
C_2     1619.544688
C_3        0.606163
C_4     3763.056161
           ...     
R_32       1.153534
R_33       1.122841
R_34       1.261200
R_35       0.995900
R_36       1.151398
Length: 148, dtype: float64

In [1]:
#!/usr/bin/python
# -*- coding: utf-8 -*-

import os
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import pdist, squareform
from pymatgen.core import Structure
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer

# ==========================================
# 0. 配置路径
# ==========================================
INPUT_FILE = "/home/gaotianjiao/modnet-master/ext_struc/feature_pervoskite_ef_noce.csv"
OUTPUT_FILE = "/home/gaotianjiao/modnet-master/ext_struc/feature_pervoskite_ef.csv"
CIF_FOLDER = "/home/gaotianjiao/modnet-master/ext_struc/perovskite_cif"
PROP_FILE = "/home/gaotianjiao/modnet-master/ext_struc/element_properties/ElementsProperties.xlsx"

# ==========================================
# 1. 核心特征计算逻辑
# ==========================================

def get_element_symbol(species_str):
    return "".join([c for c in species_str if c.isalpha()])

def compute_CE_feature(center_atom, neighbors, elements_properties):
    center_species = center_atom['species']
    
    # 错误处理：防止表格里没有该元素
    if center_species not in elements_properties.columns:
        # 这里改为返回 None，由上层处理，防止报错中断整个流程
        return None
        
    center_prop = elements_properties[center_species]
    center_indexed = pd.Series(center_prop.values, index=[f"C_{attr}" for attr in center_prop.index])

    # 这里的权重逻辑：依然使用 1/d
    weights = np.array([1 / n['distance'] if n['distance'] != 0 else 0 for n in neighbors])
    if weights.sum() > 0:
        weights /= weights.sum()
    else:
        weights = np.zeros_like(weights)

    env_prop = pd.Series(0, index=center_prop.index)
    for n, w in zip(neighbors, weights):
        n_species = n['species']
        if n_species in elements_properties.columns:
             env_prop += elements_properties[n_species] * w
        else:
            pass

    env_indexed = pd.Series(env_prop.values, index=[f"E_{attr}" for attr in env_prop.index])

    diff_prop = (center_prop - env_prop).abs()
    diff_indexed = pd.Series(diff_prop.values, index=[f"D_{attr}" for attr in center_prop.index])

    ratio_prop = center_prop / (env_prop + 1e-6)
    ratio_indexed = pd.Series(ratio_prop.values, index=[f"R_{attr}" for attr in center_prop.index])

    CE_feature = pd.concat([center_indexed, env_indexed, diff_indexed, ratio_indexed])
    return CE_feature

def load_elements_properties(file_path):
    elements_df = pd.read_excel(file_path, sheet_name=0, index_col=None, header=0)
    elements_list = pd.read_excel(file_path, sheet_name=1, index_col=None, header=0)
    elements = list(elements_list["element"])
    
    valid_elements = [e for e in elements if e in elements_df.columns]
    df = elements_df.loc[:, valid_elements]
    
    df.dropna(axis=0, how='any', inplace=True)
    df = df.reset_index(drop=True)
    property_names = elements_df["Properties"].tolist()
    return df, property_names

def get_motifs_by_shells(structure, num_shells=3, tolerance=0.01):
    try:
        sga = SpacegroupAnalyzer(structure)
        symmetrized_structure = sga.get_symmetrized_structure()
    except Exception:
        # 如果对称性分析失败，回退到使用原始结构（虽然计算量大，但保证不崩）
        symmetrized_structure = structure
        # 如果使用原始结构，需要构造一个假的 equivalent_indices
        # 这里为了简化，直接抛出异常或仅处理不对称部分，通常 sga 很少失败
        return []

    motifs = []
    
    # 这里的逻辑是只取不等价位点
    indices_to_iter = symmetrized_structure.equivalent_indices if hasattr(symmetrized_structure, "equivalent_indices") else [[i] for i in range(len(structure))]

    for i, group in enumerate(indices_to_iter):
        center_index = group[0]
        center_site = structure[center_index]

        all_neighbors = structure.get_neighbors(center_site, r=10.0)
        
        if not all_neighbors:
            continue
            
        all_neighbors.sort(key=lambda x: x[1]) 
        
        shells = []
        current_shell = []
        
        if all_neighbors:
            current_dist = all_neighbors[0][1]
            for neighbor in all_neighbors:
                site, dist = neighbor[0], neighbor[1]
                if dist - current_dist > tolerance:
                    shells.append(current_shell)
                    current_shell = []
                    current_dist = dist
                current_shell.append({
                    'species': get_element_symbol(site.specie.symbol), # 确保清洗符号
                    'distance': dist,
                    'index': neighbor[2]
                })
            if current_shell:
                shells.append(current_shell)
        
        selected_shells = shells[:num_shells]
        final_neighbors = [n for shell in selected_shells for n in shell]
        
        motifs.append({
            'central_atom': {
                'species': get_element_symbol(center_site.specie.symbol),
                'index': center_index
            },
            'neighbors': final_neighbors,
            'coordination_environment': f"{len(selected_shells)}_shells",
            'distorted': False 
        })
        
    return motifs

# [优化]：传入已加载的 properties df，而不是文件路径
def extract_CE_features_optimized(structure, elements_properties, num_shells=3):
    motifs = get_motifs_by_shells(structure, num_shells=num_shells)
    features = []

    for motif in motifs:
        center_atom = motif['central_atom']
        CE_vec = compute_CE_feature(center_atom=center_atom,
                                    neighbors=motif['neighbors'],
                                    elements_properties=elements_properties)
        if CE_vec is not None:
            features.append({
                'center_index': center_atom['index'],
                'center_species': center_atom['species'],
                'CE_feature': CE_vec
            })
    
    return features

def weighted_CE_vector(ce_features, method='hybrid', scale_method='zscore'):
    if not ce_features:
        return None
        
    all_keys = ce_features[0]['CE_feature'].index
    X = np.array([f['CE_feature'].values for f in ce_features])

    n_motifs = len(ce_features)
    if n_motifs == 1:
        return ce_features[0]['CE_feature']

    if scale_method == 'zscore':
        scaler = StandardScaler()
        # 处理全0列导致的标准差为0的警告/错误
        X_scaled = scaler.fit_transform(X)
        X_scaled = np.nan_to_num(X_scaled) # 将可能的NaN转为0
    elif scale_method == 'minmax':
        from sklearn.preprocessing import MinMaxScaler
        scaler = MinMaxScaler()
        X_scaled = scaler.fit_transform(X)
    else:
        raise ValueError("scale_method error")

    var_weights = np.var(X_scaled, axis=1)
    if np.all(var_weights == 0):
        var_weights = np.ones_like(var_weights)

    # pdist 在样本很少时可能会有问题，加个保护
    try:
        dist_matrix = squareform(pdist(X_scaled, metric='euclidean'))
        dist_weights = np.mean(dist_matrix, axis=1)
    except Exception:
        dist_weights = np.ones_like(var_weights)

    if method == 'variance':
        weights = var_weights
    elif method == 'distance':
        weights = dist_weights
    elif method == 'hybrid':
        weights = (var_weights + dist_weights) / 2
    else:
        weights = np.ones_like(var_weights)

    weights = np.maximum(weights, 1e-8)
    weights /= np.sum(weights)

    weighted_X = X * weights[:, np.newaxis]
    final_vector_values = np.sum(weighted_X, axis=0)

    final_vector = pd.Series(final_vector_values, index=all_keys)
    return final_vector

def rename_columns_list(property_names):
    """生成最终的列名列表"""
    C_cols = ["C_" + name for name in property_names]
    E_cols = ["E_" + name for name in property_names]
    D_cols = ["D_" + name for name in property_names] 
    R_cols = ["R_" + name for name in property_names]
    return C_cols + E_cols + D_cols + R_cols

# ==========================================
# 2. 批量处理主逻辑
# ==========================================

def batch_process():
    print(f"1. 加载输入文件: {INPUT_FILE}")
    df_input = pd.read_csv(INPUT_FILE)
    
    # 检查是否有 cif_file 列
    cif_col_name = 'cif_file' # 假设列名是 cif_file
    if cif_col_name not in df_input.columns:
        # 尝试自动寻找包含 .cif 的列
        potential_cols = [c for c in df_input.columns if df_input[c].astype(str).str.contains('.cif').any()]
        if potential_cols:
            cif_col_name = potential_cols[0]
            print(f"   自动检测到 CIF 文件名列: {cif_col_name}")
        else:
            raise ValueError(f"输入 CSV 中找不到 CIF 文件名列，请确认列名是否为 '{cif_col_name}'")

    print(f"2. 加载元素属性库: {PROP_FILE}")
    elements_properties_df, property_names = load_elements_properties(PROP_FILE)
    
    # 预生成列名，用于对齐
    final_cols = rename_columns_list(property_names)
    
    new_features_list = []
    failed_cifs = []

    print("3. 开始批量计算特征...")
    for index, row in tqdm(df_input.iterrows(), total=len(df_input)):
        cif_filename = row[cif_col_name]
        cif_path = os.path.join(CIF_FOLDER, cif_filename)
        
        # 初始化全 NaN 的结果，如果失败则填入 NaN
        empty_result = {col: np.nan for col in final_cols}
        
        if not os.path.exists(cif_path):
            # print(f"文件不存在: {cif_path}")
            failed_cifs.append(cif_filename)
            new_features_list.append(empty_result)
            continue

        try:
            # 加载结构
            structure = Structure.from_file(cif_path)
            
            # 提取 Motifs 特征 (使用 Shell 逻辑)
            # 注意：直接传入已加载的 df，避免重复 IO
            motif_features = extract_CE_features_optimized(
                structure=structure, 
                elements_properties=elements_properties_df, 
                num_shells=3
            )
            
            # 加权聚合
            final_vec_series = weighted_CE_vector(motif_features, method='hybrid')
            
            if final_vec_series is not None:
                # 确保列名对应
                # weighted_CE_vector 返回的 index 是 C_AtomicNumber 这种格式
                # 我们需要将其转为 dict 加入列表
                feature_dict = final_vec_series.to_dict()
                new_features_list.append(feature_dict)
            else:
                new_features_list.append(empty_result)
                
        except Exception as e:
            # print(f"处理 {cif_filename} 失败: {e}")
            failed_cifs.append(cif_filename)
            new_features_list.append(empty_result)

    print(f"4. 计算完成。失败文件数: {len(failed_cifs)}")
    if len(failed_cifs) > 0:
        print(f"   失败示例: {failed_cifs[:5]}")

    # 合并数据
    print("5. 合并并保存结果...")
    df_features = pd.DataFrame(new_features_list)

    for col in final_cols:
        if col not in df_features.columns:
            df_features[col] = np.nan
            
    # 按预定顺序重排
    df_features = df_features[final_cols]
    
    # 横向拼接：原始特征 + 新的 CE 特征
    df_final = pd.concat([df_input, df_features], axis=1)
    
    df_final.to_csv(OUTPUT_FILE, index=False)
    print(f"   保存成功: {OUTPUT_FILE}")

if __name__ == "__main__":
    batch_process()

1. 加载输入文件: /home/gaotianjiao/modnet-master/ext_struc/feature_pervoskite_ef_noce.csv
2. 加载元素属性库: /home/gaotianjiao/modnet-master/ext_struc/element_properties/ElementsProperties.xlsx
3. 开始批量计算特征...


 10%|█         | 82/810 [00:03<00:39, 18.37it/s]/home/gaotianjiao/anaconda3/lib/python3.8/site-packages/pymatgen/io/cif.py:1168: UserWarning: Issues encountered while parsing CIF: Some fractional coordinates rounded to ideal values to avoid issues with finite precision.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))
 91%|█████████ | 734/810 [00:52<00:02, 31.35it/s]/home/gaotianjiao/anaconda3/lib/python3.8/site-packages/pymatgen/io/cif.py:1168: UserWarning: Issues encountered while parsing CIF: Some fractional coordinates rounded to ideal values to avoid issues with finite precision.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))
100%|██████████| 810/810 [00:56<00:00, 14.41it/s]


4. 计算完成。失败文件数: 177
   失败示例: ['mp-981250.cif', 'mp-866816.cif', 'mp-866565.cif', 'mp-866471.cif', 'mp-866101.cif']
5. 合并并保存结果...
   保存成功: /home/gaotianjiao/modnet-master/ext_struc/feature_pervoskite_ef.csv
